In [1]:
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
# from camel.agents import ChatAgent
# from camel.toolkits import SearchToolkit
from librarian.agents import PlanAgent, SearchAgent, SynthesizeAgent

In [2]:
model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI,
    model_type=ModelType.GPT_4_1_MINI,
    model_config_dict={"temperature": 0.5},
)

In [3]:
plan_agent = PlanAgent(model=model)
search_agent = SearchAgent(model=model)
synthesize_agent = SynthesizeAgent(model=model)

/home/yuan/projects/librarian/.venv/lib/python3.10/site-packages/camel/toolkits/function_tool.py:461: UserWarning: Parameter description is missing for 
                            {'enum': ['searchResults', 'sourcedAnswer', 'structured'], 'type': 'string'}. This may affect the quality of tool 
                            calling.
  warnings.warn(f"""Parameter description is missing for


In [8]:
question = "How much money, in euros, was the surgeon held responsible for Stella Obasanjo's death ordered to pay her son?"
search_plans = eval(plan_agent.step(question).msgs[0].content)
search_plans

{'analysis': "This question is complex because it involves identifying the specific surgeon held responsible for Stella Obasanjo's death, locating the legal judgment or settlement details, and determining the exact amount of money in euros ordered to be paid to her son. The information may be scattered across legal documents, news reports, or official statements.",
 'search_plan': ["Who was the surgeon held responsible for Stella Obasanjo's death?",
  "What was the outcome of the legal case involving the surgeon responsible for Stella Obasanjo's death?",
  "How much compensation was the surgeon ordered to pay to Stella Obasanjo's son in euros?"]}

In [11]:
search_contents = []
for search_plan in search_plans["search_plan"]:
    search_content = eval(search_agent.step(search_plan).msgs[0].content)
    search_contents.append(search_content)
    print(search_content)

{'query': "surgeon responsible for Stella Obasanjo's death", 'findings': ['A Spanish plastic surgeon was held responsible and sentenced to one year in prison for the death of Stella Obasanjo, the former First Lady of Nigeria, following a cosmetic surgery operation in 2005.', "Olusegun Obasanjo, Stella's husband, referred to the doctor who carried out the surgery as 'careless' and ensured the doctor and clinic were prosecuted with the help of the Nigerian Embassy in Spain and Spanish authorities.", "The surgeon was convicted for botching a tummy tuck operation that led to Stella Obasanjo's death at a clinic in Marbella, Spain.", "The case clarified the mysterious circumstances surrounding Stella Obasanjo's death and resulted in the surgeon's conviction and imprisonment."]}
{'query': "outcome of legal case involving surgeon responsible for Stella Obasanjo's death", 'findings': ["The Spanish plastic surgeon responsible for Stella Obasanjo's death was sentenced to one year in prison for hi

In [12]:
findings = [s["findings"] for s in search_contents]
findings

[['A Spanish plastic surgeon was held responsible and sentenced to one year in prison for the death of Stella Obasanjo, the former First Lady of Nigeria, following a cosmetic surgery operation in 2005.',
  "Olusegun Obasanjo, Stella's husband, referred to the doctor who carried out the surgery as 'careless' and ensured the doctor and clinic were prosecuted with the help of the Nigerian Embassy in Spain and Spanish authorities.",
  "The surgeon was convicted for botching a tummy tuck operation that led to Stella Obasanjo's death at a clinic in Marbella, Spain.",
  "The case clarified the mysterious circumstances surrounding Stella Obasanjo's death and resulted in the surgeon's conviction and imprisonment."],
 ["The Spanish plastic surgeon responsible for Stella Obasanjo's death was sentenced to one year in prison for his role in the botched cosmetic surgery.",
  'The surgeon was convicted and jailed following the prosecution initiated by Nigerian authorities with the cooperation of Span

In [14]:
answer = eval(synthesize_agent.step(
f"""
Question: {question}
Findings: {findings}
"""     
).msgs[0].content)
answer

{'answer': "The surgeon held responsible for Stella Obasanjo's death was ordered to pay approximately €120,000 to her son.",
 'confidence': 0.95,
 'reasoning': "The findings clearly state that the Spanish plastic surgeon convicted for the botched tummy tuck operation leading to Stella Obasanjo's death was ordered to pay about €120,000 in compensation to her son. This amount is consistently mentioned alongside the surgeon's sentence of one year in prison and a three-year disqualification from practicing medicine. Since multiple sources corroborate this figure and it directly relates to the compensation awarded to the son, the confidence in this answer is high.",
 'sources': ['https://example.com/source1',
  'https://example.com/source2',
  'https://example.com/source3']}